## Импорты

In [50]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option('display.width', 1000)
plt.rcParams['figure.dpi'] = 200

pd.set_option('display.max_rows', None)  # все строки
pd.set_option('display.max_columns', None)  # все колонки
pd.set_option('display.width', None)  # автоматическая ширина
pd.set_option('display.max_colwidth', None)  # полное содержимое ячеек

## Загрузка данных

In [51]:
val_df = pd.read_parquet('../datasets/joined/valid_data.parquet')
errors_df = pd.read_csv('../datasets/backward_errors_12_03.csv')

#### Создадим датафреймы с false negative (добавленные ошибки), true positive и true negative

In [52]:
val_df['is_fn'] = val_df['event_id'].isin(errors_df['event_id']).astype(int)


fn_df = val_df[val_df['is_fn'] == 1]
tp_df = val_df[(val_df['target'] == 1) & (val_df['is_fn'] == 0)]
tn_df = val_df[(val_df['target'] == 0) & (val_df['is_fn'] == 0)]

print(f"Всего операций в valid: {len(val_df)}")
print(f"FN (пропущенные фроды): {len(fn_df)}")
print(f"TP (пойманные фроды): {len(tp_df)}")
print(f"TN (чистые): {len(tn_df)}")

Всего операций в valid: 1795327
FN (пропущенные фроды): 655
TP (пойманные фроды): 601
TN (чистые): 1794071


#### Посмотрим все колонки в валидационном сете

In [53]:
print(val_df.columns.tolist())
print(f"Всего колонок: {len(val_df.columns.tolist())}")

['event_id', 'event_dttm', 'event_type_nm', 'event_desc', 'channel_indicator_type', 'channel_indicator_sub_type', 'currency_iso_cd', 'mcc_code', 'pos_cd', 'browser_language', 'battery', 'device_system_version', 'developer_tools', 'phone_voip_call_state', 'web_rdp_connection', 'compromised', 'target', 'os_category', 'tz_category', 'language_category', 'operaton_amt_is_missing', 'operaton_amt_log', 'hour', 'day_of_week', 'is_night', 'mean_operation', 'median_operation', 'user_activity', 'avg_ops_last_week', 'count_ops_last_week', 'median_ops_last_week', 'inter_quantile_range', 'ops_count_2h', 'ops_sum_2h', 'max_amt_2h', 'is_fn']
Всего колонок: 36


In [54]:
# Проверяем диапазон дат
print("Диапазон дат в valid_data:")
print(f"От: {val_df['event_dttm'].min()}")
print(f"До: {val_df['event_dttm'].max()}")


Диапазон дат в valid_data:
От: 2024-09-30 17:00:13
До: 2024-10-31 16:59:56


## Анализ

#### Создадим словарь c группами FN, TP и TN

In [55]:
groups = {
    'FN (пропущенные)': fn_df,
    'TP (пойманные)': tp_df,
    'TN (чистые)': tn_df.sample(min(10000, len(tn_df)), random_state=42)
}

#### Суммы

In [56]:
for name, df in groups.items():
    print(f"\n{name} (n = {len(df)}):")
    print(f"  Средний лог суммы: {df['operaton_amt_log'].mean():.2f}")
    print(f"  Медианный лог суммы: {df['operaton_amt_log'].median():.2f}")
    print(f"  Пропуски суммы: {df['operaton_amt_is_missing'].mean()*100:.1f}%")


FN (пропущенные) (n = 655):
  Средний лог суммы: 6.73
  Медианный лог суммы: 8.78
  Пропуски суммы: 45.6%

TP (пойманные) (n = 601):
  Средний лог суммы: 7.87
  Медианный лог суммы: 10.30
  Пропуски суммы: 46.3%

TN (чистые) (n = 10000):
  Средний лог суммы: 6.11
  Медианный лог суммы: 8.58
  Пропуски суммы: 46.1%


#### Признаки удаленного доступа

In [57]:
for name, df in groups.items():
    print(f"\n{name}:")
    
    # RDP
    rdp_mask = df['web_rdp_connection'].isin([0, 1])
    if rdp_mask.any():
        print(f"  RDP: {df.loc[rdp_mask, 'web_rdp_connection'].mean()*100:.2f}% (n = {rdp_mask.sum()})")
    
    # Developer tools
    dev_mask = df['developer_tools'].isin([0, 1])
    if dev_mask.any():
        print(f"  Developer tools: {df.loc[dev_mask, 'developer_tools'].mean()*100:.2f}% (n = {dev_mask.sum()})")
    
    # VoIP
    voip_mask = df['phone_voip_call_state'].isin([0, 1])
    if voip_mask.any():
        print(f"  VoIP: {df.loc[voip_mask, 'phone_voip_call_state'].mean()*100:.2f}% (n = {voip_mask.sum()})")
    
    # Compromised
    comp_mask = df['compromised'].isin([0, 1])
    if comp_mask.any():
        print(f"  Compromised: {df.loc[comp_mask, 'compromised'].mean()*100:.2f}% (n = {comp_mask.sum()})")


FN (пропущенные):
  RDP: 1.94% (n = 103)
  Compromised: 0.00% (n = 202)

TP (пойманные):
  RDP: 3.70% (n = 81)
  Compromised: 0.32% (n = 311)

TN (чистые):
  RDP: 2.71% (n = 1699)
  Compromised: 0.11% (n = 2724)


#### Время и активность

In [58]:
for name, df in groups.items():
    print(f"\n{name}:")
    print(f"  Ночные (0-5ч): {df['hour'].between(0,5).mean()*100:.1f}%")
    print(f"  Выходные: {df['day_of_week'].isin([5,6]).mean()*100:.1f}%")
    
    if 'user_activity' in df.columns:
        print(f"  Средняя активность: {df['user_activity'].mean():.1f}")
        print(f"  Медианная активность: {df['user_activity'].median():.1f}")


FN (пропущенные):
  Ночные (0-5ч): 10.8%
  Выходные: 30.5%
  Средняя активность: 234.9
  Медианная активность: 214.0

TP (пойманные):
  Ночные (0-5ч): 4.3%
  Выходные: 33.3%
  Средняя активность: 159.7
  Медианная активность: 133.0

TN (чистые):
  Ночные (0-5ч): 16.5%
  Выходные: 31.9%
  Средняя активность: 277.6
  Медианная активность: 266.0


#### Операции за 2 часа

In [59]:
for name, df in groups.items():
    print(f"\n{name}:")
    if 'ops_count_2h' in df.columns:
        print(f"  Среднее кол-во: {df['ops_count_2h'].mean():.1f}")
        print(f"  Медиана кол-ва: {df['ops_count_2h'].median():.1f}")
        print(f"  Макс кол-во: {df['ops_count_2h'].max():.0f}")
        print(f"  Средняя сумма: {df['ops_sum_2h'].mean():.0f}")


FN (пропущенные):
  Среднее кол-во: 4.1
  Медиана кол-ва: 2.0
  Макс кол-во: 66
  Средняя сумма: 19589823

TP (пойманные):
  Среднее кол-во: 8.1
  Медиана кол-ва: 6.0
  Макс кол-во: 54
  Средняя сумма: 40040059

TN (чистые):
  Среднее кол-во: 2.3
  Медиана кол-ва: 1.0
  Макс кол-во: 69
  Средняя сумма: 7576796


#### Категории

In [60]:
for name, df in groups.items():
    print(f"\n{name}:")
    
    if 'os_category' in df.columns:
        print("\n  OS категории:")
        os_dist = df['os_category'].value_counts().sort_index()
        for cat, count in os_dist.items():
            pct = count / len(df) * 100
            print(f"    {cat}: {count} ({pct:.1f}%)")
    
    if 'language_category' in df.columns:
        print("\n  Языковые категории:")
        lang_dist = df['language_category'].value_counts().sort_index()
        for cat, count in lang_dist.items():
            pct = count / len(df) * 100
            print(f"    {cat}: {count} ({pct:.1f}%)")
    
    if 'tz_category' in df.columns:
        print("\n  Таймзона категории:")
        tz_dist = df['tz_category'].value_counts().sort_index()
        for cat, count in tz_dist.items():
            pct = count / len(df) * 100
            print(f"    {cat}: {count} ({pct:.1f}%)")


FN (пропущенные):

  OS категории:
    -1: 618 (94.4%)
    0: 11 (1.7%)
    1: 14 (2.1%)
    2: 5 (0.8%)
    3: 7 (1.1%)

  Языковые категории:
    -1: 617 (94.2%)
    0: 16 (2.4%)
    1: 10 (1.5%)
    2: 12 (1.8%)

  Таймзона категории:
    0: 20 (3.1%)
    1: 15 (2.3%)
    2: 620 (94.7%)

TP (пойманные):

  OS категории:
    -1: 578 (96.2%)
    0: 10 (1.7%)
    1: 9 (1.5%)
    2: 2 (0.3%)
    3: 2 (0.3%)

  Языковые категории:
    -1: 572 (95.2%)
    0: 16 (2.7%)
    1: 11 (1.8%)
    2: 2 (0.3%)

  Таймзона категории:
    0: 9 (1.5%)
    1: 8 (1.3%)
    2: 584 (97.2%)

TN (чистые):

  OS категории:
    -1: 9251 (92.5%)
    0: 595 (5.9%)
    1: 41 (0.4%)
    2: 96 (1.0%)
    3: 17 (0.2%)

  Языковые категории:
    -1: 8926 (89.3%)
    0: 847 (8.5%)
    1: 106 (1.1%)
    2: 109 (1.1%)
    3: 12 (0.1%)

  Таймзона категории:
    0: 562 (5.6%)
    1: 224 (2.2%)
    2: 9214 (92.1%)


#### MCC-коды

In [61]:
print("=== ТОП-10 MCC КОДОВ ===\n")

for name, df in groups.items():
    print(f"\n{name}:")
    # Топ-10 самых частых MCC
    mcc_top = df['mcc_code'].value_counts().head(10)
    for mcc, count in mcc_top.items():
        pct = count / len(df) * 100
        print(f"  {mcc}: {pct:.1f}% ({count} оп.)")
    
    # Доля остальных кодов
    other_count = len(df) - mcc_top.sum()
    other_pct = other_count / len(df) * 100
    print(f"  ... остальные: {other_pct:.1f}% ({other_count} оп.)")

=== ТОП-10 MCC КОДОВ ===


FN (пропущенные):
  16: 7.9% (52 оп.)
  4: 5.6% (37 оп.)
  10: 4.9% (32 оп.)
  3: 3.2% (21 оп.)
  14: 1.5% (10 оп.)
  19: 1.2% (8 оп.)
  18: 1.1% (7 оп.)
  17: 0.9% (6 оп.)
  15: 0.8% (5 оп.)
  1: 0.6% (4 оп.)
  ... остальные: 72.2% (473 оп.)

TP (пойманные):
  10: 10.0% (60 оп.)
  16: 2.0% (12 оп.)
  19: 1.2% (7 оп.)
  17: 0.8% (5 оп.)
  4: 0.3% (2 оп.)
  7: 0.3% (2 оп.)
  15: 0.3% (2 оп.)
  0: 0.2% (1 оп.)
  13: 0.2% (1 оп.)
  ... остальные: 84.7% (509 оп.)

TN (чистые):
  4: 14.1% (1412 оп.)
  10: 3.5% (348 оп.)
  15: 3.2% (325 оп.)
  17: 2.3% (233 оп.)
  1: 1.1% (111 оп.)
  19: 1.1% (106 оп.)
  9: 1.1% (106 оп.)
  7: 1.0% (103 оп.)
  14: 0.8% (80 оп.)
  16: 0.5% (51 оп.)
  ... остальные: 71.2% (7125 оп.)
